
---

# Phase 5 – Monitoring & Optimization (Production-Grade Observability)

## Overview

In a real data warehouse environment, monitoring is essential to ensure:

* Data pipelines run successfully
* Queries perform efficiently
* Compute resources are not overloaded
* Costs are controlled
* Failures are detected quickly

This project implements **production-grade monitoring in Snowflake** using **system metadata views** provided by Snowflake.

These system views expose operational metrics about:

* warehouse utilization
* query execution
* credit consumption
* pipeline health

Monitoring queries are stored in the **`monitoring` schema** and implemented as **views** so they can be easily connected to dashboards.

---

# Monitoring Architecture

The monitoring layer sits **on top of the entire pipeline**.

```
Source CSV Files
       │
       ▼
RAW Layer
       │
       ▼
STAGING Layer
       │
       ▼
PRODUCTION Layer
       │
       ▼
REPORTING Views
       │
       ▼
MONITORING & OBSERVABILITY
```

The monitoring layer tracks **system behavior rather than business data**.

---

# Snowflake System Metadata Sources

Snowflake automatically records operational metrics inside special databases and schemas.

The monitoring layer retrieves metrics from:

| Source                  | Purpose                           |
| ----------------------- | --------------------------------- |
| SNOWFLAKE.ACCOUNT_USAGE | historical account-level metadata |
| INFORMATION_SCHEMA      | near real-time metadata           |
| TABLE FUNCTIONS         | warehouse usage metrics           |

These objects are **maintained automatically by Snowflake**.

No manual logging is required.

---

# 1. Warehouse Load Monitoring

## Purpose

Monitor **how busy the compute warehouse is**.

This helps detect:

* overloaded warehouses
* queued queries
* blocked workloads
* insufficient compute resources

---

## Data Source

Metrics are fetched using the table function:

```
INFORMATION_SCHEMA.WAREHOUSE_LOAD_HISTORY()
```

This Snowflake function returns **warehouse performance statistics over time**.

---

## Monitoring View

```sql
CREATE OR REPLACE VIEW monitoring.vw_warehouse_load AS
SELECT
    start_time,
    end_time,
    warehouse_name,
    avg_running,
    avg_queued_load,
    avg_queued_provisioning,
    avg_blocked
FROM TABLE(
    INFORMATION_SCHEMA.WAREHOUSE_LOAD_HISTORY(
        DATE_RANGE_START => DATEADD('day', -7, CURRENT_TIMESTAMP()),
        DATE_RANGE_END => CURRENT_TIMESTAMP(),
        WAREHOUSE_NAME => 'INGEST_WH'
    )
);
```

---

## Metrics Explained

| Column                  | Meaning                               |
| ----------------------- | ------------------------------------- |
| start_time              | start of monitoring interval          |
| end_time                | end of monitoring interval            |
| warehouse_name          | compute warehouse name                |
| avg_running             | number of queries running             |
| avg_queued_load         | queries waiting due to overload       |
| avg_queued_provisioning | queries waiting for warehouse startup |
| avg_blocked             | queries blocked by locks              |

---

## Interpretation

Example scenario:

```
avg_running = 8
avg_queued_load = 3
```

Meaning:

* 8 queries running
* 3 queries waiting because warehouse is overloaded

Possible solution:

* increase warehouse size
* enable multi-cluster scaling

---

# 2. Query Performance Monitoring

## Purpose

Track query execution performance and workload behavior.

This allows identification of:

* slow queries
* large scans
* inefficient SQL

---

## Data Source

Monitoring uses:

```
SNOWFLAKE.ACCOUNT_USAGE.QUERY_HISTORY
```

This system table records **every query executed in the account**.

Snowflake automatically logs:

* query text
* execution time
* user
* warehouse used
* data scanned
* execution status

---

## Monitoring View

```sql
CREATE OR REPLACE VIEW monitoring.vw_query_performance AS
SELECT
    query_id,
    user_name,
    warehouse_name,
    database_name,
    schema_name,
    query_text,
    execution_status,
    total_elapsed_time/1000 AS execution_seconds,
    rows_produced,
    bytes_scanned,
    start_time
FROM SNOWFLAKE.ACCOUNT_USAGE.QUERY_HISTORY
WHERE start_time >= DATEADD('day', -7, CURRENT_TIMESTAMP());
```

---

## Key Metrics

| Metric            | Meaning            |
| ----------------- | ------------------ |
| query_id          | unique identifier  |
| query_text        | SQL executed       |
| execution_seconds | total runtime      |
| bytes_scanned     | data scanned       |
| rows_produced     | result size        |
| execution_status  | success or failure |

---

## Why This Is Important

Large **bytes_scanned** indicates:

* missing clustering
* inefficient filtering
* full table scans

---

# 3. Long-Running Query Detection

## Purpose

Identify queries that take excessive time.

Slow queries often indicate:

* poor query design
* large scans
* inefficient joins

---

## Monitoring View

```sql
CREATE OR REPLACE VIEW monitoring.vw_long_running_queries AS
SELECT
    query_id,
    user_name,
    warehouse_name,
    total_elapsed_time/1000 AS execution_seconds,
    query_text,
    start_time
FROM SNOWFLAKE.ACCOUNT_USAGE.QUERY_HISTORY
WHERE total_elapsed_time > 60000
ORDER BY execution_seconds DESC;
```

---

## Threshold

```
60000 ms = 60 seconds
```

Queries longer than 60 seconds are flagged.

---

# 4. Failed Query Monitoring

## Purpose

Detect pipeline failures and SQL errors.

Failures may occur due to:

* syntax errors
* missing tables
* permission issues
* resource limits

---

## Monitoring View

```sql
CREATE OR REPLACE VIEW monitoring.vw_failed_queries AS
SELECT
    query_id,
    user_name,
    warehouse_name,
    execution_status,
    error_message,
    start_time
FROM SNOWFLAKE.ACCOUNT_USAGE.QUERY_HISTORY
WHERE execution_status != 'SUCCESS'
ORDER BY start_time DESC;
```

---

## Example Failure

Possible error message:

```
SQL compilation error: object does not exist
```

This helps quickly diagnose issues in the pipeline.

---

# 5. Warehouse Credit Usage Monitoring

## Purpose

Monitor **compute cost consumption**.

Snowflake charges credits based on warehouse usage.

Monitoring helps detect:

* inefficient workloads
* unnecessary warehouse activity
* cost spikes

---

## Data Source

```
SNOWFLAKE.ACCOUNT_USAGE.WAREHOUSE_METERING_HISTORY
```

This system table records:

* credits consumed
* warehouse activity
* time intervals

---

## Monitoring View

```sql
CREATE OR REPLACE VIEW monitoring.vw_credit_usage AS
SELECT
    warehouse_name,
    start_time,
    end_time,
    credits_used
FROM SNOWFLAKE.ACCOUNT_USAGE.WAREHOUSE_METERING_HISTORY
WHERE start_time >= DATEADD('day', -7, CURRENT_TIMESTAMP());
```

---

## Metric Explanation

| Column         | Meaning                  |
| -------------- | ------------------------ |
| warehouse_name | compute warehouse        |
| start_time     | metering start           |
| end_time       | metering end             |
| credits_used   | compute credits consumed |

---

## Why This Matters

Example:

```
credits_used = 10
```

Meaning warehouse consumed **10 Snowflake credits** during the interval.

Monitoring helps control **cloud costs**.

---

# 6. Expensive Query Detection

## Purpose

Identify queries scanning large volumes of data.

Large scans indicate:

* missing filters
* inefficient joins
* poor partitioning

---

## Monitoring View

```sql
CREATE OR REPLACE VIEW monitoring.vw_expensive_queries AS
SELECT
    query_id,
    warehouse_name,
    bytes_scanned/1024/1024/1024 AS gb_scanned,
    total_elapsed_time/1000 AS execution_seconds,
    query_text
FROM SNOWFLAKE.ACCOUNT_USAGE.QUERY_HISTORY
WHERE bytes_scanned > 1000000000
ORDER BY gb_scanned DESC;
```

---

## Threshold

```
1,000,000,000 bytes ≈ 1 GB
```

Queries scanning more than **1GB** are flagged.

---

# 7. Pipeline Data Monitoring

## Purpose

Validate that pipeline layers contain expected data.

Row counts are monitored across:

* raw tables
* staging tables
* production tables

---

## Monitoring View

```sql
CREATE OR REPLACE VIEW monitoring.vw_pipeline_row_counts AS
SELECT 'raw.customers' AS table_name, COUNT(*) AS row_count FROM raw.customers
UNION ALL
SELECT 'raw.products', COUNT(*) FROM raw.products
UNION ALL
SELECT 'raw.transactions', COUNT(*) FROM raw.transactions
UNION ALL
SELECT 'production.fact_sales', COUNT(*) FROM production.fact_sales;
```

---

# Monitoring Queries for Operations

## Warehouse Load

```sql
SELECT *
FROM monitoring.vw_warehouse_load
ORDER BY start_time DESC;
```

---

## Long Running Queries

```sql
SELECT *
FROM monitoring.vw_long_running_queries;
```

---

## Failed Queries

```sql
SELECT *
FROM monitoring.vw_failed_queries;
```

---

## Credit Usage

```sql
SELECT
warehouse_name,
SUM(credits_used)
FROM monitoring.vw_credit_usage
GROUP BY warehouse_name;
```

---

# Benefits of This Monitoring System

This monitoring layer provides:

### Operational Visibility

* warehouse load monitoring
* query execution tracking
* pipeline health checks

### Performance Optimization

* identify slow queries
* detect expensive scans
* improve SQL efficiency

### Cost Management

* monitor credit consumption
* detect warehouse inefficiencies

### Reliability

* detect failed queries
* quickly diagnose errors

---

# Final System Architecture

```
Data Sources (CSV)
        │
        ▼
RAW Layer
        │
        ▼
STAGING Layer
        │
        ▼
PRODUCTION Layer
        │
        ▼
REPORTING Views
        │
        ▼
MONITORING & OBSERVABILITY
```

The monitoring layer ensures the **data warehouse operates efficiently, reliably, and cost-effectively**.

---
